# Colocalization: SER-fallback thresholds (data processing)

Companion to the simulation threshold panel (`Data_Processing/Figure_3/FigED1AB.SER_Fallback_Threshold`, Extended Data Figure 1a,b), on the GTEx eQTL (gene × tissue) and UK Biobank blood-trait (region × trait) colocalization replication analyses with ADSP and 1000 Genomes LD. SuSiE-rRSS falls back to the SER fit when `Q_art` exceeds the artifact threshold (default 0.1) or a credible set's log-BF attenuation `DCS_max` reaches the sensitivity threshold (default 30). Both only set the flag after fitting, so each setting is re-applied to the stored per-record coloc results (`16`–`19` `[coloc]`); one threshold is varied at a time with the other at its default. `artifact = 1` / `sensitivity = Off` turns that check off. Replication and non-replication are pooled as in Fig 5a,b (`coloc_agg()` in `R/utils.R`).

**Output:** `figure_data/FigED1CD.SER_Fallback_Threshold.rds`: replication rate, non-replication rate (bootstrap 95% CI) and fraction of pairs using the SER fallback, by threshold × dataset × LD reference

In [1]:
source("../../R/utils.R")

data_name <- "FigED1CD.SER_Fallback_Threshold"
n_boot    <- 200        # bootstrap resamples per setting (Fig 5a/b use 1000)

coloc_dirs <- c(
  "GTEx eQTL|ADSP"              = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/coloc/result",
  "GTEx eQTL|1000 Genomes"      = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/coloc/result",
  "UKB blood traits|ADSP"         = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/coloc/result",
  "UKB blood traits|1000 Genomes" = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/coloc/result"
)

## per-record coloc results (read_coloc_records() in R/utils.R): SuSiE-rRSS fit + SER fit per gene x tissue
recs <- imap(coloc_dirs, function(d, nm) {
  read_coloc_records(d) %>%
    filter(method %in% c("susie_eb_mix_res", "susie_ser_res")) %>%
    mutate(dataset = str_split_fixed(nm, "\\|", 2)[, 1], reference = str_split_fixed(nm, "\\|", 2)[, 2])
})
map_int(recs, nrow)

GTEx eQTL|ADSP        GTEx eQTL|1000 Genomes 
                       107868                        105838 
        UKB blood traits|ADSP UKB blood traits|1000 Genomes 
                         8950                          9555

In [2]:
## ---- fallback rule, re-applied at any thresholds -----------------------------
## susieR: R_reliability_flag = (artifact evaluable & Q_art > artifact_threshold) |
##                              (max CS log-BF attenuation >= sensitivity_threshold)
## The coloc records keep Q_art and DCS_max but not artifact_evaluable: a finite Q_art is taken as evaluable.
fallback_flag <- function(d, artifact, sensitivity)
  (is.finite(d$Q_art) & d$Q_art > artifact) | (is.finite(d$DCS_max) & d$DCS_max >= sensitivity)

## check: the defaults (0.1, 30) reproduce the stored flag on the SuSiE-rRSS rows
map_dfr(recs, function(r) {
  eb <- filter(r, method == "susie_eb_mix_res")
  tibble(dataset = r$dataset[1], reference = r$reference[1],
         n = nrow(eb), agree = mean(fallback_flag(eb, 0.1, 30) == (eb$reliability_flag %in% TRUE)))
})

dataset,reference,n,agree
<chr>,<chr>,<int>,<dbl>
GTEx eQTL,ADSP,54175,1
GTEx eQTL,1000 Genomes,53350,1
UKB blood traits,ADSP,4504,1
UKB blood traits,1000 Genomes,4837,1


In [3]:
## ---- threshold grid: vary one, keep the other at its default ------------------
grid <- bind_rows(
  tibble(sweep = "artifact",    artifact = c(0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1),   sensitivity = 30),
  tibble(sweep = "sensitivity", artifact = 0.1, sensitivity = c(log(3), log(20), log(150), 10, 20, 30, 50, 100, Inf))
) %>%
  mutate(setting = row_number(), is_default = artifact == 0.1 & sensitivity == 30)

## SuSiE-rRSS at one setting, as add_ser_hybrids() in R/utils.R: SER row when flagged, SuSiE-rRSS row otherwise
## (decision taken on gene x tissue pairs with a SuSiE-rRSS record)
hybrid <- function(r, artifact, sensitivity) {
  dec <- r %>% filter(method == "susie_eb_mix_res") %>%
    transmute(gene, tissue, use_ser = fallback_flag(., artifact, sensitivity))
  r %>% inner_join(dec, by = c("gene", "tissue")) %>%
    filter((!use_ser & method == "susie_eb_mix_res") | (use_ser & method == "susie_ser_res"))
}

## pooled rates as coloc_agg() in R/utils.R, bootstrap over records
rates <- function(h, B) {
  ri <- h$replicated_insample; ni <- h$n_insample_cs; ro <- h$replicated_outsample; no <- h$n_outsample_cs
  ri[is.na(ri)] <- 0; ro[is.na(ro)] <- 0; ni[is.na(ni)] <- 0; no[is.na(no)] <- 0
  est <- c(rep = sum(ri) / sum(ni), nrep = 1 - sum(ro) / sum(no))
  bs <- replicate(B, { i <- sample.int(length(ri), replace = TRUE)
                       c(sum(ri[i]) / sum(ni[i]), 1 - sum(ro[i]) / sum(no[i])) })
  tibble(metric = c("Replication rate", "Non-replication rate"), est = unname(est),
         lo = apply(bs, 1, quantile, 0.025, na.rm = TRUE), hi = apply(bs, 1, quantile, 0.975, na.rm = TRUE))
}

set.seed(2026)
summ <- map_dfr(recs, function(r) {
  pmap_dfr(grid, function(sweep, artifact, sensitivity, setting, is_default) {
    h <- hybrid(r, artifact, sensitivity)
    n_dec <- n_distinct(paste(h$gene, h$tissue))
    bind_rows(rates(h, n_boot),
              tibble(metric = "SER fallback",
                     est = mean(distinct(h, gene, tissue, use_ser)$use_ser), lo = NA_real_, hi = NA_real_)) %>%
      mutate(sweep = sweep, artifact = artifact, sensitivity = sensitivity, setting = setting,
             is_default = is_default, n_pairs = n_dec)
  }) %>% mutate(dataset = r$dataset[1], reference = r$reference[1])
})

In [4]:
## ---- plot-ready ------------------------------------------------------------------
fmt_thr <- function(x) case_when(is.infinite(x) ~ "Off",
                                 abs(x - log(3))   < 1e-9 ~ "log 3",
                                 abs(x - log(20))  < 1e-9 ~ "log 20",
                                 abs(x - log(150)) < 1e-9 ~ "log 150",
                                 TRUE ~ as.character(x))

plot_df <- summ %>%
  mutate(
    threshold     = if_else(sweep == "artifact", artifact, sensitivity),
    threshold_lab = if_else(sweep == "artifact" & artifact == 1, "Off", fmt_thr(threshold)),
    sweep         = factor(sweep, levels = c("artifact", "sensitivity"),
                           labels = c("Artifact threshold (Q_art)", "Sensitivity threshold (CS log-BF attenuation)")),
    dataset       = factor(dataset, levels = c("GTEx eQTL", "UKB blood traits")),
    reference     = factor(reference, levels = c("ADSP", "1000 Genomes")),
    metric        = factor(metric, levels = c(coloc_metric_levels, "SER fallback"))
  ) %>%
  select(dataset, reference, sweep, threshold, threshold_lab, is_default, metric, est, lo, hi, n_pairs)

## default setting: should equal SuSiE-rRSS in Fig 5a,b / S10a,b
plot_df %>% filter(is_default, sweep == levels(sweep)[1]) %>% select(dataset, reference, metric, est)

save_data(plot_df, data_name)

dataset,reference,metric,est
<fct>,<fct>,<fct>,<dbl>
GTEx eQTL,ADSP,Replication rate,0.77330076
GTEx eQTL,ADSP,Non-replication rate,0.04343748
GTEx eQTL,ADSP,SER fallback,0.01028700
GTEx eQTL,1000 Genomes,Replication rate,0.72346712
GTEx eQTL,1000 Genomes,Non-replication rate,0.04169023
GTEx eQTL,1000 Genomes,SER fallback,0.21620451
UKB blood traits,ADSP,Replication rate,0.80768689
UKB blood traits,ADSP,Non-replication rate,0.03940309
UKB blood traits,ADSP,SER fallback,0.03181313
